In [ ]:
# Métricas: BLEU, BERTScore, MoverScore, METEOR e ROUGE

import sacrebleu
import nltk
import pandas as pd

from rouge_score import rouge_scorer
from bert_score import score as bert_score
from nltk.translate.meteor_score import meteor_score
from moverscore_v2 import get_idf_dict, word_mover_score
rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

# Fazendo um df exemplo para testar

In [9]:
df = pd.read_csv('C:/PokeAI/databases/Perguntas_Metagame.csv')
df['resposta_gerada'] = df['resposta']
df

,pergunta,resposta,resposta_gerada
0,Qual é o Pokémon com maior taxa de uso absolut...,Kingambit é o Pokémon dominante absoluto do fo...,Kingambit é o Pokémon dominante absoluto do fo...
1,Qual é a taxa de uso do Mega Charizard Y no Re...,"Mega Charizard Y possui uma taxa de uso de 50,...","Mega Charizard Y possui uma taxa de uso de 50,..."
2,Como o desempenho e a viabilidade do Sneasler ...,Sneasler decaiu ligeiramente de S-tier no Regu...,Sneasler decaiu ligeiramente de S-tier no Regu...
3,Por que Garchomp é classificado no tier S de v...,"Garchomp possui 45,00% de uso em Bo3 devido à ...","Garchomp possui 45,00% de uso em Bo3 devido à ..."
4,Qual é a taxa de uso e a principal função do I...,"Incineroar possui uma taxa de uso de 42,30% em...","Incineroar possui uma taxa de uso de 42,30% em..."
...,...,...,...
116,Quais são os três Pokémon com maior taxa de te...,Os três maiores parceiros do Farigiraf são Syl...,Os três maiores parceiros do Farigiraf são Syl...
117,Com quais parceiros de equipe o suporte Mausho...,Maushold se associa com maior frequência a Sin...,Maushold se associa com maior frequência a Sin...
118,Com qual Pokémon o atacante físico Mega Metagr...,Mega Metagross tem sua maior associação de equ...,Mega Metagross tem sua maior associação de equ...
119,Qual a taxa de associação de equipe entre Mega...,"Mega Aerodactyl compartilha 61,31% de teammate...","Mega Aerodactyl compartilha 61,31% de teammate..."


# Gerando todas métricas automáticas

In [ ]:
referencias = df['resposta'].fillna('').astype(str).tolist()
predicoes = df['resposta_gerada'].fillna('').astype(str).tolist()

# BERTScore
precisao, recall, f1 = bert_score(predicoes, referencias, lang="pt", verbose=False, batch_size=8)
df['bertscore_precision'] = precisao.tolist()
df['bertscore_recall'] = recall.tolist()
df['bertscore_f1'] = f1.tolist()

idf_ref = get_idf_dict(referencias)
idf_pred = get_idf_dict(predicoes)

# MoverScore
notas_moverscore = word_mover_score(referencias, predicoes, idf_ref, idf_pred, stop_words=[], n_gram=1, remove_subwords=True)
df['moverscore'] = notas_moverscore

for indice, row in df.iterrows():
    referencia = row['resposta']
    predicao = row['resposta_gerada']

    # ROUGE
    notas_rouge = rouge.score(referencia, predicao)

    # BLEU
    nota_bleu = sacrebleu.sentence_bleu(predicao, [referencia]).score

    # METEOR
    nota_meteor = meteor_score([referencia.split()], predicao.split())

    df.at[indice, 'rougeL'] = notas_rouge['rougeL'].fmeasure
    df.at[indice, 'rouge1'] = notas_rouge['rouge1'].fmeasure
    df.at[indice, 'rouge2'] = notas_rouge['rouge2'].fmeasure
    df.at[indice, 'bleu'] = nota_bleu
    df.at[indice, 'meteor'] = nota_meteor

In [8]:
df

,pergunta,resposta,resposta_gerada,bertscore_precision,bertscore_recall,bertscore_f1,moverscore,rougeL,rouge1,rouge2,bleu,meteor
0,Qual é o Pokémon com maior taxa de uso absolut...,Kingambit é o Pokémon dominante absoluto do fo...,Kingambit é o Pokémon dominante absoluto do fo...,1.0,1.0,1.0,0.999581,1.0,1.0,1.0,100.0,0.999964
1,Qual é a taxa de uso do Mega Charizard Y no Re...,"Mega Charizard Y possui uma taxa de uso de 50,...","Mega Charizard Y possui uma taxa de uso de 50,...",1.0,1.0,1.0,0.999664,1.0,1.0,1.0,100.0,0.999995
2,Como o desempenho e a viabilidade do Sneasler ...,Sneasler decaiu ligeiramente de S-tier no Regu...,Sneasler decaiu ligeiramente de S-tier no Regu...,1.0,1.0,1.0,0.999600,1.0,1.0,1.0,100.0,0.999996
3,Por que Garchomp é classificado no tier S de v...,"Garchomp possui 45,00% de uso em Bo3 devido à ...","Garchomp possui 45,00% de uso em Bo3 devido à ...",1.0,1.0,1.0,0.999737,1.0,1.0,1.0,100.0,0.999995
4,Qual é a taxa de uso e a principal função do I...,"Incineroar possui uma taxa de uso de 42,30% em...","Incineroar possui uma taxa de uso de 42,30% em...",1.0,1.0,1.0,0.999702,1.0,1.0,1.0,100.0,0.999993
...,...,...,...,...,...,...,...,...,...,...,...,...
116,Quais são os três Pokémon com maior taxa de te...,Os três maiores parceiros do Farigiraf são Syl...,Os três maiores parceiros do Farigiraf são Syl...,1.0,1.0,1.0,0.999787,1.0,1.0,1.0,100.0,0.999852
117,Com quais parceiros de equipe o suporte Mausho...,Maushold se associa com maior frequência a Sin...,Maushold se associa com maior frequência a Sin...,1.0,1.0,1.0,0.999712,1.0,1.0,1.0,100.0,0.999953
118,Com qual Pokémon o atacante físico Mega Metagr...,Mega Metagross tem sua maior associação de equ...,Mega Metagross tem sua maior associação de equ...,1.0,1.0,1.0,0.999655,1.0,1.0,1.0,100.0,0.999818
119,Qual a taxa de associação de equipe entre Mega...,"Mega Aerodactyl compartilha 61,31% de teammate...","Mega Aerodactyl compartilha 61,31% de teammate...",1.0,1.0,1.0,0.999685,1.0,1.0,1.0,100.0,0.999953
